# Customer Churn - Model Training

Train Logistic Regression and XGBoost models to predict customer churn. Compare performance and save the best model.

In [ ]:
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Paths
DATA_PATH = os.path.join("..", "data", "customers.csv")
MODEL_PATH = os.path.join("..", "data", "churn_model.joblib")

# Feature configuration
NUMERIC_FEATURES = ["age", "tenure", "monthly_charges", "total_charges"]
CATEGORICAL_FEATURES = ["contract_type", "payment_method"]
TARGET = "churn"

## 1. Load & Prepare Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop_duplicates(subset="customer_id")

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]

print(f"Features: {list(X.columns)}")
print(f"Target distribution:\n{y.value_counts(normalize=True).round(3)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain size: {len(X_train):,} | Test size: {len(X_test):,}")

## 2. Build Preprocessing Pipeline

In [ ]:
def build_preprocessor():
    """Create preprocessing pipeline for numeric and categorical features."""
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ]
    )

preprocessor = build_preprocessor()
print("Preprocessor configured for numeric scaling and one-hot encoding.")

## 3. Train Models

In [ ]:
# Logistic Regression
lr_pipeline = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_pipeline.predict_proba(X_test)[:, 1])

print("Logistic Regression Results:")
print(f"  Accuracy: {lr_acc:.4f}")
print(f"  ROC-AUC : {lr_auc:.4f}")
print(f"  Confusion Matrix:\n{confusion_matrix(y_test, lr_pred)}")
print(classification_report(y_test, lr_pred, target_names=["Retained", "Churned"]))

In [ ]:
# XGBoost
xgb_pipeline = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("classifier", XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss",
    )),
])

xgb_pipeline.fit(X_train, y_train)
xgb_pred = xgb_pipeline.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_pipeline.predict_proba(X_test)[:, 1])

print("XGBoost Results:")
print(f"  Accuracy: {xgb_acc:.4f}")
print(f"  ROC-AUC : {xgb_auc:.4f}")
print(f"  Confusion Matrix:\n{confusion_matrix(y_test, xgb_pred)}")
print(classification_report(y_test, xgb_pred, target_names=["Retained", "Churned"]))

## 4. Model Comparison

In [ ]:
# Compare models side by side
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost"],
    "Accuracy": [lr_acc, xgb_acc],
    "ROC-AUC": [lr_auc, xgb_auc],
})
print("Model Comparison:")
print(comparison.to_string(index=False))

# Confusion matrix heatmaps
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, name, pred in [
    (axes[0], "Logistic Regression", lr_pred),
    (axes[1], "XGBoost", xgb_pred),
]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=ax,
        xticklabels=["Retained", "Churned"],
        yticklabels=["Retained", "Churned"],
    )
    ax.set_title(f"{name} - Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
# ROC curves with Plotly
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_pipeline.predict_proba(X_test)[:, 1])
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_pipeline.predict_proba(X_test)[:, 1])

fig = go.Figure()
fig.add_trace(go.Scatter(x=lr_fpr, y=lr_tpr, mode="lines", name=f"Logistic Regression (AUC={lr_auc:.3f})"))
fig.add_trace(go.Scatter(x=xgb_fpr, y=xgb_tpr, mode="lines", name=f"XGBoost (AUC={xgb_auc:.3f})"))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random", line=dict(dash="dash")))
fig.update_layout(
    title="ROC Curve Comparison",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
)
fig.show()

## 5. Feature Importance (XGBoost)

In [ ]:
# Extract feature names after preprocessing
preprocessor_fitted = xgb_pipeline.named_steps["preprocessor"]
cat_encoder = preprocessor_fitted.named_transformers_["cat"]
feature_names = NUMERIC_FEATURES + list(cat_encoder.get_feature_names_out(CATEGORICAL_FEATURES))

# XGBoost feature importances
importances = xgb_pipeline.named_steps["classifier"].feature_importances_
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values("importance", ascending=True)

# Plotly horizontal bar chart
fig = px.bar(
    importance_df,
    x="importance",
    y="feature",
    orientation="h",
    title="XGBoost Feature Importance",
    color="importance",
    color_continuous_scale="Viridis",
)
fig.update_layout(template="plotly_white", height=450)
fig.show()

print("\nTop Features:")
print(importance_df.sort_values("importance", ascending=False).head(5).to_string(index=False))

## 6. Save Best Model

In [ ]:
# Select best model by accuracy
models = {
    "LogisticRegression": (lr_pipeline, lr_acc),
    "XGBoost": (xgb_pipeline, xgb_acc),
}

best_name = max(models, key=lambda k: models[k][1])
best_model, best_acc = models[best_name]

joblib.dump(best_model, MODEL_PATH)
print(f"Best model: {best_name}")
print(f"Accuracy  : {best_acc:.4f}")
print(f"Saved to  : {MODEL_PATH}")